# Flash Flash Revolution Bronze Layer API Ingestion

This notebook ingests Flash Flash Revolution game data from the FFR API and builds bronze layer tables in Unity Catalog. It supports **incremental updates** by detecting changes via `swf_version` tracking, minimizes API calls through change detection, and uses parallel processing for efficient chart data retrieval.

## Pipeline Overview

**Data Sources:**
- FFR Songlist API (`api.php?action=songlist`) - Song metadata with version tracking
- FFR Chart API (`api.php?action=chartFFR`) - Note patterns for individual songs
- FFR Playlist API (`r3-playlist.php`) - Complete song attributes and flags

**Output Tables:**
- `acubed.ffr.bronze__songlist` - Song metadata with `swf_version` for change detection
- `acubed.ffr.bronze__charts` - Chart note patterns with nested arrays of note sequences
- `acubed.ffr.bronze__playlist` - Full song attributes including flags, ratings, and release dates

---

## Incremental Processing Logic

**Change Detection:**
1. Compare current API response against existing bronze__songlist table
2. Detect three types of changes:
   - **New songs**: Present in API but not in bronze table
   - **Updated songs**: `swf_version` differs between API and bronze table
   - **Deleted songs**: Present in bronze table but not in API response
3. Only fetch chart data for changed songs (minimizes API load)

**Delete Handling:**
- Songs removed from API are automatically deleted from bronze tables
- Uses SQL DELETE statements before updating tables
- Maintains referential consistency across songlist, charts, and playlist tables

**Version Tracking:**
- `swf_version` uniquely identifies each song's content state
- Upstream chart updates trigger new version numbers
- Enables precise change detection without comparing full payloads

---

## Parallel Chart Fetching

**Implementation:**
- ThreadPoolExecutor with configurable pool size
- Connection pooling via requests.Session
- Retry logic with exponential backoff
- Timeout protection per request

**Why Parallel:**
- Chart API requires individual requests per song_id
- Sequential fetching is prohibitively slow for full refreshes
- Thread pool balances speed with API rate limits

---

## Configuration

**API Authentication:**
- API key stored in Databricks secret scope `ffr`
- Retrieved via `dbutils.secrets.get(scope='ffr', key='api-key')`

**Parallel Processing:**
- `THREAD_POOL_SIZE`: Number of concurrent API requests
- `MAX_RETRIES`: Retry attempts for failed requests
- `REQUEST_TIMEOUT`: Timeout per API call (seconds)

---

## Table Schemas

**bronze__songlist:**
- `id` (bigint): Unique song identifier
- `name` (string): Song title
- `difficulty` (int): Official difficulty rating
- `note_count` (int): Total notes in chart
- `swf_version` (long): Version identifier for change detection

**bronze__charts:**
- `song_id` (bigint): Foreign key to songlist
- `notes` (array): Nested array structure of note sequences

**bronze__playlist:**
- `level` (int): Song ID (matches songlist.id)
- `name` (string): Song title
- `difficulty` (int): Difficulty rating
- `genre` (string): Genre classification
- `time` (string): Song duration
- Additional columns: flags, ratings, release dates, author info

In [0]:
# Import libraries
import requests
import json
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from concurrent.futures import ThreadPoolExecutor
from operator import itemgetter
from delta.tables import DeltaTable

# Configuration
API_KEY = dbutils.secrets.get(scope='ffr', key='api-key')
BASE_API_URL = "https://www.flashflashrevolution.com/api/api.php"
PLAYLIST_URL = "https://www.flashflashrevolution.com/game/r3/r3-playlist.php"

# Table names
TABLE_SONGLIST = "acubed.ffr.bronze__songlist"
TABLE_CHARTS = "acubed.ffr.bronze__charts"
TABLE_PLAYLIST = "acubed.ffr.bronze__playlist"

# Parallel processing configuration
THREAD_POOL_SIZE = 4
MAX_RETRIES = 3
REQUEST_TIMEOUT = 10

print("✓ Configuration loaded successfully")

In [0]:

# Step 1: Capture previous state if table exists
previous_df = None
table_exists = spark.catalog.tableExists(TABLE_SONGLIST)

if table_exists:
    previous_df = spark.table(TABLE_SONGLIST)
    print(f"✓ Captured previous state: {previous_df.count()} songs")
else:
    print(f"ℹ No previous table found (first run - will create new table)")

# Step 2: Fetch data from API
params = {"key": API_KEY, "action": "songlist"}
response = requests.get(BASE_API_URL, params=params)
songs_data = response.json()

# Convert to Spark DataFrame
df = spark.createDataFrame(songs_data)
print(f"\n✓ Fetched {df.count()} songs from API")

# Step 3: Detect and handle DELETED songs BEFORE updating table
if previous_df is not None:
    # Detect deleted songs (in previous but not in new API response)
    deleted_songs = previous_df.alias("old").join(
        df.alias("new"),
        col("old.id") == col("new.id"),
        "left_anti"
    ).select(
        col("old.id").alias("song_id"),
        col("old.name").alias("song_name")
    )
    
    deleted_count = deleted_songs.count()
    
    if deleted_count > 0:
        print(f"\n⚠ Found {deleted_count} deleted songs (removed from API source)")
        display(deleted_songs.orderBy("song_id"))
        
        # Delete from songlist table
        deleted_ids = [row.song_id for row in deleted_songs.collect()]
        deleted_ids_str = ','.join([str(id) for id in deleted_ids])
        spark.sql(f"DELETE FROM {TABLE_SONGLIST} WHERE id IN ({deleted_ids_str})")
        print(f"✓ Deleted {deleted_count} songs from {TABLE_SONGLIST}")
        
        # Delete corresponding charts from bronze__charts table
        if spark.catalog.tableExists(TABLE_CHARTS):
            charts_before = spark.table(TABLE_CHARTS).count()
            spark.sql(f"DELETE FROM {TABLE_CHARTS} WHERE song_id IN ({deleted_ids_str})")
            charts_after = spark.table(TABLE_CHARTS).count()
            charts_deleted = charts_before - charts_after
            print(f"✓ Deleted {charts_deleted} chart records from {TABLE_CHARTS}")
    else:
        print(f"\n✓ No deleted songs detected")

# Step 4: Detect changes and new songs (BEFORE updating table)
changed_ids = []  # Initialize empty list

if previous_df is not None:
    # Detect songs with swf_version changes (INNER JOIN)
    version_changes = previous_df.alias("old").join(
        df.alias("new"),
        col("old.id") == col("new.id"),
        "inner"
    ).filter(
        col("old.swf_version") != col("new.swf_version")
    ).select(
        col("new.id").alias("song_id"),
        col("new.name").alias("song_name"),
        col("old.swf_version").alias("previous_swf_version"),
        col("new.swf_version").alias("new_swf_version")
    )
    
    # Detect new songs (LEFT ANTI JOIN - songs in new but not in old)
    new_songs = df.alias("new").join(
        previous_df.alias("old"),
        col("new.id") == col("old.id"),
        "left_anti"
    ).select(
        col("new.id").alias("song_id"),
        col("new.name").alias("song_name")
    )
    
    version_changes_count = version_changes.count()
    new_songs_count = new_songs.count()
    
    print(f"\n{'='*60}")
    print(f"CHANGE DETECTION SUMMARY")
    print(f"{'='*60}")
    print(f"Songs with swf_version changes: {version_changes_count}")
    print(f"New songs added: {new_songs_count}")
    print(f"Total songs to update: {version_changes_count + new_songs_count}")
    print(f"{'='*60}")
    
    # Display version changes if any
    if version_changes_count > 0:
        print("\nℹ Songs with swf_version changes:")
        display(version_changes.orderBy("song_id"))
    
    # Display new songs if any
    if new_songs_count > 0:
        print("\nℹ Newly added songs:")
        display(new_songs.orderBy("song_id"))
    
    # Combine both changed and new song IDs
    if version_changes_count > 0 or new_songs_count > 0:
        changed_song_ids = [row.song_id for row in version_changes.select("song_id").collect()]
        new_song_ids = [row.song_id for row in new_songs.select("song_id").collect()]
        changed_ids = changed_song_ids + new_song_ids
        print(f"\n✓ Extracted {len(changed_ids)} song IDs for chart update ({len(changed_song_ids)} changed + {len(new_song_ids)} new)")
    else:
        print("\n✓ No changes or new songs detected - no chart updates needed")
else:
    print("\nℹ First run: All songs marked for chart fetching")
    # On first run, mark all songs for chart fetching
    changed_ids = [row.id for row in df.select("id").collect()]
    print(f"✓ All {len(changed_ids)} songs will be fetched in next cell")

# Step 5: Write to Delta table (merge for incremental, overwrite for first run)
if table_exists:
    # Incremental update: merge new and changed songs
    print(f"\nℹ Performing incremental update (merge)...")
    delta_table = DeltaTable.forName(spark, TABLE_SONGLIST)
    
    delta_table.alias("target").merge(
        df.alias("source"),
        "target.id = source.id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    
    final_count = spark.table(TABLE_SONGLIST).count()
    print(f"✓ Merged updates into {TABLE_SONGLIST} (now {final_count} songs)")
else:
    # First run: create table with all songs
    print(f"\nℹ First run: Creating table with all songs...")
    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(TABLE_SONGLIST)
    
    print(f"✓ Created table {TABLE_SONGLIST} with {df.count()} songs")

# Display sample of current data
print("\nCurrent table sample:")
display(spark.table(TABLE_SONGLIST).limit(10))

In [0]:
# Determine which songs to fetch
try:
    # Check if changed_ids exists from previous cell
    if 'changed_ids' in globals():
        if len(changed_ids) > 0:
            print(f"ℹ Incremental mode: Fetching charts for {len(changed_ids)} changed songs")
            songs_to_fetch = changed_ids
        else:
            print("✓ No changes detected - skipping chart fetch")
            if spark.catalog.tableExists(TABLE_CHARTS):
                print("Current bronze__charts table:")
                display(spark.table(TABLE_CHARTS).limit(5))
            else:
                print("ℹ bronze__charts table does not exist yet (first run)")
            songs_to_fetch = []  # Empty list to skip fetch
    else:
        print("ℹ Full refresh mode: Fetching charts for all songs")
        songs_df = spark.table(TABLE_SONGLIST)
        songs_to_fetch = [row.id for row in songs_df.select("id").collect()]
except NameError:
    print("ℹ Full refresh mode: Fetching charts for all songs")
    songs_df = spark.table(TABLE_SONGLIST)
    songs_to_fetch = [row.id for row in songs_df.select("id").collect()]

# Only proceed with fetch if there are songs to fetch
if len(songs_to_fetch) > 0:
    # Configure session with connection pooling
    session = requests.Session()
    session.mount(
        "https://",
        requests.adapters.HTTPAdapter(
            pool_maxsize=THREAD_POOL_SIZE,
            max_retries=MAX_RETRIES,
            pool_block=True,
        ),
    )

    total_songs = len(songs_to_fetch)
    print(f"Fetching chart data for {total_songs} songs using {THREAD_POOL_SIZE} threads...")

    # Build request URLs
    songs_with_urls = [
        {
            "song_id": song_id,
            "url": f"{BASE_API_URL}?key={API_KEY}&action=chart&level={song_id}"
        }
        for song_id in songs_to_fetch
    ]

    def fetch_chart(url):
        """Fetch chart data from API."""
        try:
            response = session.get(url, timeout=REQUEST_TIMEOUT)
            return response
        except Exception as e:
            print(f"Error fetching {url}: {e}")
            return None

    # Fetch chart data in parallel
    chart_data = []
    with ThreadPoolExecutor(max_workers=THREAD_POOL_SIZE) as executor:
        for song_info, response in zip(
            songs_with_urls,
            executor.map(fetch_chart, map(itemgetter("url"), songs_with_urls)),
        ):
            if response and response.status_code == 200:
                response_json = response.json()
                chart_data.append({
                    "song_id": song_info["song_id"],
                    "info": response_json.get("info"),
                    "chart": response_json.get("chart")
                })
            
            # Progress indicator
            if len(chart_data) % 100 == 0:
                print(f"Progress: {len(chart_data)}/{total_songs} songs processed")

    print(f"Successfully fetched chart data for {len(chart_data)} songs")

    # Create DataFrame from chart data
    chart_df = spark.createDataFrame(chart_data)

    # Check if table exists for merge/upsert logic
    table_exists = spark.catalog.tableExists(TABLE_CHARTS)

    if table_exists and 'changed_ids' in globals() and len(changed_ids) > 0:
        # Incremental update: merge new data into existing table
        print(f"\nℹ Performing incremental update (merge)...")
        
        delta_table = DeltaTable.forName(spark, TABLE_CHARTS)
        
        delta_table.alias("target").merge(
            chart_df.alias("source"),
            "target.song_id = source.song_id"
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()
        
        print(f"✓ Merged {chart_df.count()} records into {TABLE_CHARTS}")
    else:
        # Full refresh: overwrite entire table
        print(f"\nℹ Performing full refresh (overwrite)...")
        chart_df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(TABLE_CHARTS)
        
        print(f"✓ Created table {TABLE_CHARTS} with {chart_df.count()} records")

    display(spark.table(TABLE_CHARTS).limit(5))
else:
    print("✓ Skipping chart fetch - no songs to process")

In [0]:
print(f"Fetching playlist data from {PLAYLIST_URL}...")

# Use User-Agent header to mimic browser request
headers = {
    "User-Agent": " ".join([
        "Mozilla/5.0 (Windows NT 6.1; WOW64)",
        "AppleWebKit/537.36 (KHTML, like Gecko)",
        "Chrome/56.0.2924.76",
        "Safari/537.36",
    ])
}

response = requests.get(PLAYLIST_URL, headers=headers, timeout=REQUEST_TIMEOUT)

if response.status_code == 200:
    playlist_data = response.json()
    print(f"Successfully fetched {len(playlist_data)} songs from playlist")
    
    # Convert to pandas first to handle mixed types better
    pandas_df = pd.DataFrame(playlist_data)
    
    # Convert all numeric columns to consistent types
    for col in pandas_df.columns:
        if pandas_df[col].dtype in ['float64', 'int64']:
            # Convert to float to handle mixed int/float
            pandas_df[col] = pandas_df[col].astype(float)
    
    # Convert to Spark DataFrame
    playlist_df = spark.createDataFrame(pandas_df)
    
    # Write to Delta table
    playlist_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(TABLE_PLAYLIST)
    
    print(f"✓ Created table {TABLE_PLAYLIST} with {playlist_df.count()} records")
    
    # Display sample data
    print("\nSample data:")
    display(spark.table(TABLE_PLAYLIST).limit(10))
else:
    print(f"Error: Failed to fetch playlist (status code: {response.status_code})")
    print(f"Response: {response.text[:500]}")